In [ ]:
"""
Lottery Ticket Hypothesis - CIFAR-10
======================================
Paper: "The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks"
Authors: Frankle & Carlin (MIT, 2019)

This script implements Iterative Magnitude Pruning (IMP) on a CNN trained on CIFAR-10.
The process:
  1. Initialize and save original weights
  2. Train the full network
  3. Prune the p% lowest-magnitude weights (globally)
  4. Reset surviving weights to their ORIGINAL initialization values
  5. Retrain the sparse network
  6. Repeat steps 3-5 for multiple pruning rounds

Instead of training for 60 epochs for example, here we train first for 10 epochs, then
prune 20% of the remaining weights, reset surviving weights to their original init values, and retrain for another 10 epochs.
So every 10 epochs, the training will be faster and faster as the network gets sparser, and we can see how the accuracy evolves 
as we prune more and more weights. Getting a sparse subnetwork that can match or exceed the original dense network's accuracy 
is what the Lottery Ticket Hypothesis is all about!
"""

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE    = 128
EPOCHS        = 8          # epochs per pruning round (increase for better accuracy)
LR            = 0.001
#MOMENTUM      = 0.9
WEIGHT_DECAY  = 1e-4
PRUNE_PERCENT = 20          # % of REMAINING weights to prune each round
NUM_ROUNDS    = 4           # pruning rounds  (after round k, sparsity ≈ 1-(0.8^k)), total epochs = EPOCHS * (NUM_ROUNDS + 1)
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Using device: {DEVICE}")


# ─────────────────────────────────────────────
# Data
# ─────────────────────────────────────────────
def get_dataloaders():
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    train_set = torchvision.datasets.CIFAR10(root="./data", train=True,
                                             download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(root="./data", train=False,
                                             download=True, transform=test_transform)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                               shuffle=True,  num_workers=2)
    test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=BATCH_SIZE,
                                               shuffle=False, num_workers=2)
    return train_loader, test_loader


# ─────────────────────────────────────────────
# Model  (simple CNN, similar to original paper)
# ─────────────────────────────────────────────
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)
        self.fc1   = nn.Linear(128 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 256)
        self.fc3   = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))   # 32→16
        x = F.relu(self.conv3(x))
        x = self.pool(F.relu(self.conv4(x)))   # 16→8
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x


# ─────────────────────────────────────────────
# Pruning utilities
# ─────────────────────────────────────────────
def get_prunable_params(model):
    """Return list of (name, weight_tensor) for all Conv/Linear layers."""
    params = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            params.append((name + ".weight", module.weight))
    return params


def compute_global_threshold(model, mask_dict, prune_percent):
    """
    Compute the magnitude threshold for global unstructured pruning.
    Only considers currently un-pruned weights (mask == 1).
    It will stablish a treshold to prune the lowest-magnitude weights across the entire network, 
    based on the specified percentage of remaining weights to prune.
    """
    all_weights = []
    for name, param in get_prunable_params(model):
        mask = mask_dict[name]
        # collect absolute values of weights that are still alive
        all_weights.append(param.data[mask.bool()].abs().cpu())
    all_weights = torch.cat(all_weights)
    threshold = torch.quantile(all_weights, prune_percent / 100.0).item()
    return threshold


def apply_mask(model, mask_dict):
    """Zero-out pruned weights by applying the binary mask in-place.
    
    This ensures that pruned weights remain zero during training.
    We do this by multiplying the weight tensor by the mask (which has 0s for pruned weights and 1s for alive weights).
    """
    for name, param in get_prunable_params(model):
        param.data *= mask_dict[name].to(param.device)


def prune_by_magnitude(model, mask_dict, prune_percent):
    """
    Update mask_dict: set mask=0 for lowest-magnitude weights
    (PRUNE_PERCENT % of currently alive weights, globally).
    """
    threshold = compute_global_threshold(model, mask_dict, prune_percent)
    for name, param in get_prunable_params(model):
        alive      = mask_dict[name].bool()
        to_prune   = (param.data.abs() <= threshold) & alive
        mask_dict[name][to_prune] = 0
    return mask_dict


def reset_to_init(model, init_state, mask_dict):
    """
    Reset surviving weights to their ORIGINAL initialization values.
    This is the key step that separates LTH from ordinary pruning!
    """
    current_state = model.state_dict()
    for name, param in get_prunable_params(model):
        mask = mask_dict[name]
        # put original init values back, then re-apply mask
        current_state[name] = init_state[name].clone() * mask.to(init_state[name].device)
    model.load_state_dict(current_state)


def sparsity(mask_dict):
    total = sum(m.numel() for m in mask_dict.values())
    zeros = sum((m == 0).sum().item() for m in mask_dict.values())
    return zeros / total * 100


# ─────────────────────────────────────────────
# Train / Evaluate
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, mask_dict):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = F.cross_entropy(outputs, targets)
        loss.backward()
        # ← zero-out gradients of pruned weights so they stay zero
        apply_mask(model, mask_dict)
        optimizer.step()
        apply_mask(model, mask_dict)   # make sure mask holds after step
        total_loss += loss.item() * inputs.size(0)
        correct    += outputs.argmax(1).eq(targets).sum().item()
        total      += inputs.size(0)
    return total_loss / total, correct / total * 100


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = model(inputs)
        correct += outputs.argmax(1).eq(targets).sum().item()
        total   += inputs.size(0)
    return correct / total * 100


def train_model(model, train_loader, test_loader, mask_dict, epochs, round_label):
    optimizer = optim.Adam(model.parameters(), lr=LR,
                          weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_acc  = 0.0

    print(f"\n{'─'*55}")
    print(f"  {round_label}  |  Sparsity: {sparsity(mask_dict):.1f}%")
    print(f"{'─'*55}")
    print(f"  {'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Test Acc':>8}")
    print(f"{'─'*55}")

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, mask_dict)
        te_acc          = evaluate(model, test_loader)
        scheduler.step()
        best_acc = max(best_acc, te_acc)
        print(f"  {epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>8.2f}%  {te_acc:>7.2f}%")

    print(f"{'─'*55}")
    print(f"  Best test accuracy: {best_acc:.2f}%")
    return best_acc


# ─────────────────────────────────────────────
# Main experiment
# ─────────────────────────────────────────────
def main():
    train_loader, test_loader = get_dataloaders()

    # ── Build model and save ORIGINAL initialization ──
    model = SimpleCNN().to(DEVICE)
    init_state = copy.deepcopy(model.state_dict())   # θ₀  ← never changes

    # ── Build initial mask (all ones = nothing pruned) ──
    mask_dict = {}
    for name, param in get_prunable_params(model):
        mask_dict[name] = torch.ones_like(param.data)

    results = []   # (round, sparsity%, best_accuracy)

    # ── Round 0: train the full (dense) network ──
    acc = train_model(model, train_loader, test_loader,
                      mask_dict, EPOCHS, "Round 0 — Dense network")
    results.append((0, 0.0, acc))

    # ── Iterative pruning rounds ──
    for rnd in range(1, NUM_ROUNDS + 1):
        # 1. Prune
        mask_dict = prune_by_magnitude(model, mask_dict, PRUNE_PERCENT)
        sp        = sparsity(mask_dict)

        # 2. Reset surviving weights to ORIGINAL init values  ← the LTH magic
        reset_to_init(model, init_state, mask_dict)

        # 3. Retrain the sparse "winning ticket"
        acc = train_model(model, train_loader, test_loader,
                          mask_dict, EPOCHS, f"Round {rnd} — Pruning round")
        results.append((rnd, sp, acc))

    # ─────────────────────────────────────────────
    # Results summary
    # ─────────────────────────────────────────────
    print("\n" + "═"*55)
    print("  RESULTS SUMMARY")
    print("═"*55)
    print(f"  {'Round':>5}  {'Sparsity':>9}  {'Weights kept':>12}  {'Test Acc':>9}")
    print("─"*55)
    for rnd, sp, acc in results:
        kept = 100 - sp
        print(f"  {rnd:>5}  {sp:>8.1f}%  {kept:>11.1f}%  {acc:>8.2f}%")
    print("═"*55)

    # ─────────────────────────────────────────────
    # Plot
    # ─────────────────────────────────────────────
    rounds      = [r[0] for r in results]
    sparsities  = [r[1] for r in results]
    accuracies  = [r[2] for r in results]
    weights_kept = [100 - s for s in sparsities]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Lottery Ticket Hypothesis — CIFAR-10", fontsize=14, fontweight="bold")

    # Left: accuracy vs pruning round
    ax1.plot(rounds, accuracies, "o-", color="#2196F3", linewidth=2, markersize=8)
    ax1.axhline(accuracies[0], color="gray", linestyle="--", label=f"Dense baseline ({accuracies[0]:.1f}%)")
    ax1.set_xlabel("Pruning round")
    ax1.set_ylabel("Test accuracy (%)")
    ax1.set_title("Accuracy across pruning rounds")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Right: accuracy vs % weights remaining
    ax2.plot(weights_kept, accuracies, "s-", color="#E91E63", linewidth=2, markersize=8)
    for wk, acc in zip(weights_kept, accuracies):
        ax2.annotate(f"{acc:.1f}%", (wk, acc), textcoords="offset points",
                     xytext=(5, 5), fontsize=8)
    ax2.axhline(accuracies[0], color="gray", linestyle="--", label=f"Dense baseline ({accuracies[0]:.1f}%)")
    ax2.set_xlabel("Weights remaining (%)")
    ax2.set_ylabel("Test accuracy (%)")
    ax2.set_title("Accuracy vs network sparsity")
    ax2.invert_xaxis()
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("lottery_ticket_results.png", dpi=150, bbox_inches="tight")
    print("\nPlot saved as 'lottery_ticket_results.png'")
    plt.show()


if __name__ == "__main__":
    main()

Using device: cpu


d:\Pablo_Data\Documentos\VSCode\deep-learning\venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



───────────────────────────────────────────────────────
  Round 0 — Dense network  |  Sparsity: 0.0%
───────────────────────────────────────────────────────
  Epoch  Train Loss  Train Acc  Test Acc
───────────────────────────────────────────────────────
      1      1.8621     29.03%    45.49%
      2      1.4616     46.12%    58.42%
      3      1.2283     55.74%    65.33%
      4      1.0779     61.62%    67.94%
      5      0.9970     65.08%    71.87%
      6      0.9285     67.77%    72.44%
      7      0.8800     69.48%    74.38%
      8      0.8338     71.28%    75.70%
      9      0.7927     72.77%    75.72%
     10      0.7607     73.69%    76.27%
     11      0.7307     74.76%    78.07%
     12      0.7112     75.55%    78.80%
     13      0.6917     76.13%    79.35%
     14      0.6796     76.67%    79.39%
     15      0.6695     76.97%    79.40%
───────────────────────────────────────────────────────
  Best test accuracy: 79.40%

────────────────────────────────────────────

KeyboardInterrupt: 